# Model Training — Coffee Bean Quality Detection
### Flat 4-class screening: 10 model kandidat dari `docs/modeling-strategy.md`

Notebook ini melatih ke-10 model yang direkomendasikan di
[`docs/modeling-strategy.md`](https://github.com/Ardiyanto24/coffee-bean-quality-detection/blob/main/docs/modeling-strategy.md)
memakai `dataset_preprocessed/` (hasil `CBQD - Preprocessing.ipynb`, sudah di R2/DVC) dan
membandingkan hasilnya di satu held-out test set yang sama.

**Skema label: flat 4-kelas** (`defect`/`longberry`/`peaberry`/`premium`) — model #7 & #8
punya 2 stage/head internal (tipe bean × status rusak), tapi output akhirnya tetap dipetakan
balik ke salah satu dari 4 label flat itu untuk dievaluasi, supaya tetap sebanding
head-to-head dengan 8 model lainnya.

**Protokol data:**
- `dataset_preprocessed/train/` (929 gambar, `cv_fold` 0-3) → `cv_fold==0` jadi **validation**
  (~232), `cv_fold∈{1,2,3}` jadi **fit** (~697) — 1 split screening, bukan full 4-fold CV
  (kelayakan waktu untuk 10 model sekaligus).
- `dataset_preprocessed/test/` (231 gambar) → held-out **test**, dipakai SEKALI di akhir tiap
  model untuk laporan, tidak untuk tuning.
- `dataset_preprocessed/real_world/` (200 gambar, unlabeled) → **tidak dipakai sama sekali**
  di notebook ini.

**PENTING — DRY_RUN:** notebook ini dikontrol oleh satu flag `DRY_RUN` di Section 2. Jalankan
dulu dengan `DRY_RUN=True` (epoch kecil) untuk memastikan seluruh kode jalan tanpa error dari
awal sampai akhir, baru ubah ke `DRY_RUN=False` untuk run penuh (50 epoch).

**Catatan GPU:** TIDAK pip install/upgrade `torch`/`torchvision` — dipakai versi bawaan
Kaggle yang sudah dipasangkan dengan benar ke GPU yang dialokasikan sesi ini (pelajaran dari
notebook FiftyOne: reinstall torch bisa menarik build CUDA yang tidak cocok GPU yang didapat).

## Section 1 — Environment & Data Provenance Setup

In [ ]:
# Sub-Step 1.1
# Tujuan: Install dependency tambahan (tanpa menyentuh torch/torchvision)

!pip install -q lightgbm timm

import os, json
from pathlib import Path

GIT_REPO_URL = "https://github.com/Ardiyanto24/coffee-bean-quality-detection.git"
PROJECT_DIR = "/kaggle/working/coffee-bean-quality-detection"
if not os.path.exists(PROJECT_DIR):
    os.system(f"git clone {GIT_REPO_URL} {PROJECT_DIR}")
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

In [ ]:
# Sub-Step 1.2
# Tujuan: Konfigurasi kredensial R2 (private dataset jika ada, fallback ke Kaggle Secrets)

# Lokasi mount dataset di /kaggle/input pernah berubah antar kernel (kadang
# /kaggle/input/<slug>/, kadang /kaggle/input/datasets/<slug>/) -- cari filenya
# lewat rglob supaya tidak bergantung pada satu struktur path yang diasumsikan.
_matches = list(Path("/kaggle/input").rglob("r2_credentials.json")) if os.path.exists("/kaggle/input") else []
cred_path = _matches[0] if _matches else None

if cred_path is not None:
    creds = json.loads(cred_path.read_text())
    os.environ["AWS_ACCESS_KEY_ID"] = creds["R2_ACCESS_KEY_ID"]
    os.environ["AWS_SECRET_ACCESS_KEY"] = creds["R2_SECRET_ACCESS_KEY"]
    print("Kredensial R2 dimuat dari private Kaggle Dataset (nilai tidak di-print).")
else:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["AWS_ACCESS_KEY_ID"] = secrets.get_secret("R2_ACCESS_KEY_ID")
    os.environ["AWS_SECRET_ACCESS_KEY"] = secrets.get_secret("R2_SECRET_ACCESS_KEY")
    print("Kredensial R2 dimuat dari Kaggle Secrets (nilai tidak di-print).")

In [ ]:
# Sub-Step 1.3
# Tujuan: Tarik dataset mentah + hasil preprocessing dari R2 (dvc pull)

!pip install -q "dvc[s3]"
!dvc pull dataset dataset_preprocessed metadata -v
print("dataset/ ada:", Path("dataset").exists())
print("dataset_preprocessed/ ada:", Path("dataset_preprocessed").exists())
print("manifest_preprocessed.csv ada:", Path("metadata/manifest_preprocessed.csv").exists())

## Section 2 — Konfigurasi Eksperimen

`DRY_RUN=True` -> epoch kecil untuk tes kode (lihat rancangan di plan). Ubah ke `False`
untuk run penuh (50 epoch, patience 10) SETELAH dry-run terbukti jalan bersih dari awal
sampai akhir.

In [ ]:
# Sub-Step 2.1
# Tujuan: Flag DRY_RUN + turunan epoch/patience

import random
import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DRY_RUN = True  # <-- ubah ke False untuk full run, SETELAH dry-run lolos bersih

if DRY_RUN:
    EPOCHS_PHASE1 = 1
    EPOCHS_PHASE2 = 4
    EARLY_STOP_PATIENCE = 2
    SCHEDULER_PATIENCE = 1
else:
    EPOCHS_PHASE1 = 5
    EPOCHS_PHASE2 = 45
    EARLY_STOP_PATIENCE = 10
    SCHEDULER_PATIENCE = 4

BATCH_SIZE = 32
IMG_SIZE = 224
LR_PHASE1 = 1e-3
LR_PHASE2 = 3e-4
LR_PHASE2_VIT = 5e-5
WEIGHT_DECAY = 0.01
CLASS_NAMES = ["defect", "longberry", "peaberry", "premium"]
LABEL_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DRY_RUN={DRY_RUN} | device={device} | epochs phase1/phase2={EPOCHS_PHASE1}/{EPOCHS_PHASE2} | patience={EARLY_STOP_PATIENCE}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Section 3 — Data: Manifest, Dataset, Transform, DataLoader

Augmentasi (train only) dinyatakan sebagai rentang -- disampel ulang secara acak tiap kali
sebuah gambar dipakai (bukan transformasi tetap). Lihat rancangan untuk tabel lengkap
rentang & alasannya. Validation/test hanya resize+normalize, tidak pernah diaugmentasi.

In [ ]:
# Sub-Step 3.1
# Tujuan: Load manifest, definisikan fit/val/test DataFrame

import pandas as pd

manifest = pd.read_csv("metadata/manifest_preprocessed.csv")
PREP_DIR = Path("dataset_preprocessed")
RAW_DIR = Path("dataset")

train_pool = manifest[manifest["split"] == "train"].reset_index(drop=True)
test_df = manifest[manifest["split"] == "test"].reset_index(drop=True)

fit_df = train_pool[train_pool["cv_fold"].isin([1, 2, 3])].reset_index(drop=True)
val_df = train_pool[train_pool["cv_fold"] == 0].reset_index(drop=True)

print(f"fit={len(fit_df)}  val={len(val_df)}  test={len(test_df)}")
print(fit_df["label"].value_counts())

In [ ]:
# Sub-Step 3.2
# Tujuan: Dataset & transform (augmentasi rentang untuk train, resize+normalize untuk val/test)

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=180),                    # sudut acak dari -180 s/d +180
    T.RandomAffine(degrees=0, translate=(0.1, 0.1)),   # translasi acak -10% s/d +10%
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.02),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class BeanDataset(Dataset):
    """Dataset flat 4-class (default) atau label custom lewat `label_fn(label_str) -> int`
    (dipakai model #7 untuk sub-task 3-kelas tipe / biner rusak-tidak). `weights` opsional
    dipakai model noise-robust (#9); default semua 1.0 (tidak berpengaruh ke model lain)."""

    def __init__(self, df, root_dir, transform, weights=None, label_fn=None):
        self.paths = [root_dir / p for p in df["image_path"]]
        label_fn = label_fn if label_fn is not None else (lambda l: LABEL_TO_IDX[l])
        self.labels = [label_fn(l) for l in df["label"]]
        self.transform = transform
        self.weights = weights if weights is not None else [1.0] * len(df)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transform(img)
        return img, self.labels[idx], self.weights[idx]


class MultiTaskDataset(Dataset):
    """Untuk model #8: mengembalikan (image, damage_label 0/1, type_label 0-2 atau -1
    untuk defect, weight). type_label=-1 berarti "abaikan saat hitung loss_type"."""

    TYPE_MAP = {"premium": 0, "peaberry": 1, "longberry": 2}

    def __init__(self, df, root_dir, transform):
        self.paths = [root_dir / p for p in df["image_path"]]
        self.damage_labels = [1 if l == "defect" else 0 for l in df["label"]]
        self.type_labels = [self.TYPE_MAP.get(l, -1) for l in df["label"]]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transform(img)
        return img, self.damage_labels[idx], self.type_labels[idx]


fit_ds = BeanDataset(fit_df, PREP_DIR, train_transform)
val_ds = BeanDataset(val_df, PREP_DIR, eval_transform)
test_ds = BeanDataset(test_df, PREP_DIR, eval_transform)

fit_loader = DataLoader(fit_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print("DataLoaders siap.")

## Section 4 — Fungsi Utilitas Bersama (dipakai semua model CNN)

In [ ]:
# Sub-Step 4.1
# Tujuan: evaluate(): macro-F1, accuracy, per-class report, confusion matrix

import torch.nn as nn
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

@torch.no_grad()
def evaluate(model, loader, device, combine_fn=None, class_names=None):
    """class_names default ke CLASS_NAMES global (4-kelas flat). Model #7/#8 memakai
    class_names 3-kelas (tipe) atau 2-kelas (rusak/tidak) saat melatih sub-model secara
    individual, sebelum digabung jadi prediksi 4-kelas lewat combine_fn.
    combine_fn opsional: dipakai untuk menggabungkan output mentah jadi prediksi 4-kelas
    flat sebelum dibandingkan ke ground truth (dipakai di evaluasi akhir #7/#8)."""
    class_names = class_names if class_names is not None else CLASS_NAMES
    model.eval()
    all_preds, all_labels = [], []
    for images, labels, _ in loader:
        images = images.to(device)
        outputs = model(images)
        preds = combine_fn(outputs) if combine_fn is not None else outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds if isinstance(preds, list) else preds.tolist())
        all_labels.extend(labels.tolist())
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    acc = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=class_names,
                                    output_dict=True, zero_division=0)
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(class_names))))
    return {"macro_f1": macro_f1, "accuracy": acc, "report": report, "confusion_matrix": cm,
            "y_true": all_labels, "y_pred": all_preds}


TYPE_TO_FLAT = {0: LABEL_TO_IDX["premium"], 1: LABEL_TO_IDX["peaberry"], 2: LABEL_TO_IDX["longberry"]}


@torch.no_grad()
def evaluate_combined(predict_fn, loader, device):
    """predict_fn(images_tensor_on_device) -> array prediksi 4-kelas flat. Dipakai model
    #7 (gabungan 2 model damage+type) dan #8 (gabungan 2 head dari 1 model)."""
    all_preds, all_labels = [], []
    for images, labels, _ in loader:
        images = images.to(device)
        preds = predict_fn(images)
        all_preds.extend(list(preds))
        all_labels.extend(labels.tolist())
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    acc = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES,
                                    output_dict=True, zero_division=0)
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(CLASS_NAMES))))
    return {"macro_f1": macro_f1, "accuracy": acc, "report": report, "confusion_matrix": cm,
            "y_true": all_labels, "y_pred": all_preds}

In [ ]:
# Sub-Step 4.2
# Tujuan: set_backbone_frozen() & train_one_model(): loop 2-fase + early stopping

import copy
import time


def set_backbone_frozen(model, head_module, frozen: bool):
    head_param_ids = set(id(p) for p in head_module.parameters())
    for p in model.parameters():
        p.requires_grad = (id(p) in head_param_ids) or (not frozen)


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for images, labels, weights in loader:
        images, labels, weights = images.to(device), labels.to(device), weights.to(device).float()
        optimizer.zero_grad()
        outputs = model(images)
        per_sample_loss = criterion(outputs, labels)
        loss = (per_sample_loss * weights).mean() if per_sample_loss.dim() > 0 else per_sample_loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
    return total_loss / len(loader.dataset)


def train_one_model(model, head_module, fit_loader, val_loader, device, model_name,
                     lr_phase2=LR_PHASE2, criterion=None, combine_fn=None, class_names=None):
    model = model.to(device)
    criterion = criterion if criterion is not None else nn.CrossEntropyLoss(reduction="none")
    best_state, best_val_f1, patience_counter = None, -1.0, 0
    history = []
    t0 = time.time()

    # ---- Phase 1: head-only ----
    set_backbone_frozen(model, head_module, frozen=True)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=LR_PHASE1, weight_decay=WEIGHT_DECAY
    )
    for epoch in range(EPOCHS_PHASE1):
        train_loss = train_epoch(model, fit_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, val_loader, device, combine_fn=combine_fn, class_names=class_names)
        history.append({"phase": 1, "epoch": epoch, "train_loss": train_loss, "val_macro_f1": val_metrics["macro_f1"]})
        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1, best_state, patience_counter = val_metrics["macro_f1"], copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1

    # ---- Phase 2: full fine-tune ----
    set_backbone_frozen(model, head_module, frozen=False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr_phase2, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=SCHEDULER_PATIENCE)
    patience_counter = 0
    for epoch in range(EPOCHS_PHASE2):
        train_loss = train_epoch(model, fit_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, val_loader, device, combine_fn=combine_fn, class_names=class_names)
        scheduler.step(val_metrics["macro_f1"])
        history.append({"phase": 2, "epoch": epoch, "train_loss": train_loss, "val_macro_f1": val_metrics["macro_f1"]})
        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1, best_state, patience_counter = val_metrics["macro_f1"], copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"  [{model_name}] early stop di phase-2 epoch {epoch} (val_macro_f1 terbaik={best_val_f1:.4f})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    elapsed = time.time() - t0
    print(f"[{model_name}] selesai dalam {elapsed/60:.1f} menit, val_macro_f1 terbaik={best_val_f1:.4f}")
    return model, history, best_val_f1

In [ ]:
# Sub-Step 4.3
# Tujuan: results store (inkremental) supaya kegagalan satu model tidak menghapus hasil sebelumnya

import os

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
results = {}
cnn_val_f1 = {}     # model_name -> val macro-F1, dipakai model #10 memilih CNN terbaik
cnn_test_proba = {}  # model_name -> softmax probabilities di test set (n_test, 4)


@torch.no_grad()
def predict_proba_torch(model, loader, device):
    model.eval()
    all_proba = []
    for images, _, _ in loader:
        images = images.to(device)
        proba = torch.softmax(model(images), dim=1).cpu().numpy()
        all_proba.append(proba)
    return np.concatenate(all_proba, axis=0)


def save_result(name, test_metrics, val_f1, extra=None):
    entry = {
        "model": name,
        "val_macro_f1": val_f1,
        "test_macro_f1": test_metrics["macro_f1"],
        "test_accuracy": test_metrics["accuracy"],
    }
    for cls in CLASS_NAMES:
        entry[f"test_recall_{cls}"] = test_metrics["report"][cls]["recall"]
    if extra:
        entry.update(extra)
    results[name] = entry
    pd.DataFrame(results.values()).to_csv(RESULTS_DIR / "model_comparison_partial.csv", index=False)
    print(f"Hasil '{name}' disimpan. Total model selesai: {len(results)}")
    return entry

In [ ]:
# Sub-Step 4.4
# Tujuan: build_model(): factory arsitektur torchvision/timm -> (model, head_module)

from torchvision import models as tv_models


def build_model(arch, num_classes):
    """Mengembalikan (model, head_module). head_module dipakai set_backbone_frozen()
    untuk tahu parameter mana yang tetap trainable saat backbone dibekukan di fase 1."""
    if arch == "mobilenet_v3_large":
        m = tv_models.mobilenet_v3_large(weights=tv_models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        in_f = m.classifier[-1].in_features
        m.classifier[-1] = nn.Linear(in_f, num_classes)
        head = m.classifier[-1]
    elif arch == "efficientnet_b0":
        m = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_f = m.classifier[-1].in_features
        m.classifier[-1] = nn.Linear(in_f, num_classes)
        head = m.classifier[-1]
    elif arch == "resnet18":
        m = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1)
        in_f = m.fc.in_features
        m.fc = nn.Linear(in_f, num_classes)
        head = m.fc
    elif arch == "convnext_tiny":
        m = tv_models.convnext_tiny(weights=tv_models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        in_f = m.classifier[-1].in_features
        m.classifier[-1] = nn.Linear(in_f, num_classes)
        head = m.classifier[-1]
    elif arch == "deit_tiny":
        import timm
        m = timm.create_model("deit_tiny_patch16_224", pretrained=True, num_classes=num_classes)
        head = m.get_classifier()
    else:
        raise ValueError(f"Arsitektur tidak dikenal: {arch}")
    return m, head


class EfficientNetMultiTask(nn.Module):
    """Untuk model #8: satu backbone EfficientNet-B0 + 2 head terpisah."""

    def __init__(self):
        super().__init__()
        backbone = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_f = backbone.classifier[-1].in_features
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.head_damage = nn.Linear(in_f, 2)
        self.head_type = nn.Linear(in_f, 3)

    def forward(self, x):
        feats = self.backbone(x)
        return self.head_damage(feats), self.head_type(feats)

## Model 1 — Gradient Boosting (LightGBM) di atas 11 fitur hand-crafted

Fitur bentuk+warna+tekstur dihitung ulang dari gambar **mentah** (`orig_path`, sebelum
crop-to-bbox) karena crop membuat `area_frac`/`center_offset` kehilangan makna aslinya
(nyaris konstan setelah bean diposisikan ulang ke tengah oleh crop). Ini murni reproduksi
fitur dari EDA v2 Section 06-08.

In [ ]:
# Sub-Step M1.1
# Tujuan: Hitung 11 fitur hand-crafted dari gambar mentah untuk fit/val/test

import cv2


def handcrafted_features(path):
    bgr = cv2.imread(str(path))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    gray_f = gray.astype(np.float32)
    thresh = gray_f.mean() - 0.6 * gray_f.std()
    mask = gray_f < thresh
    h, w = gray.shape
    if mask.sum() > 0:
        ys, xs = np.where(mask)
        area_frac = mask.sum() / (h * w)
        bbox_h, bbox_w = float(ys.max() - ys.min()), float(xs.max() - xs.min())
        bbox_ratio = max(bbox_h, bbox_w) / max(min(bbox_h, bbox_w), 1e-6)
        cy, cx = ys.mean(), xs.mean()
        center_offset = float(np.hypot(cy - h / 2, cx - w / 2) / (h / 2))
    else:
        area_frac = bbox_ratio = center_offset = np.nan

    edges = cv2.Canny(gray, 100, 200)
    return {
        "mean_r": rgb[:, :, 0].mean(), "mean_g": rgb[:, :, 1].mean(), "mean_b": rgb[:, :, 2].mean(),
        "std_r": rgb[:, :, 0].std(), "std_g": rgb[:, :, 1].std(), "std_b": rgb[:, :, 2].std(),
        "edge_density": edges.mean() / 255, "variance": float(np.var(gray)),
        "area_frac": area_frac, "bbox_ratio": bbox_ratio, "center_offset": center_offset,
    }


FEATURE_COLS = ["mean_r", "mean_g", "mean_b", "std_r", "std_g", "std_b",
                "edge_density", "variance", "area_frac", "bbox_ratio", "center_offset"]


def build_feature_matrix(df):
    feats = [handcrafted_features(RAW_DIR / p) for p in df["orig_path"]]
    X = pd.DataFrame(feats)[FEATURE_COLS].values
    y = np.array([LABEL_TO_IDX[l] for l in df["label"]])
    return X, y


X_fit, y_fit = build_feature_matrix(fit_df)
X_val, y_val = build_feature_matrix(val_df)
X_test, y_test = build_feature_matrix(test_df)
print("Feature matrix shapes:", X_fit.shape, X_val.shape, X_test.shape)

In [ ]:
# Sub-Step M1.2
# Tujuan: Latih LightGBM, evaluasi di val lalu test

import lightgbm as lgb
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

lgb_model = lgb.LGBMClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8,
    random_state=SEED, verbose=-1,
)
lgb_model.fit(X_fit, y_fit)

val_pred = lgb_model.predict(X_val)
val_f1_m1 = f1_score(y_val, val_pred, average="macro")
print("Val macro-F1:", round(val_f1_m1, 4))

test_pred = lgb_model.predict(X_test)
test_metrics_m1 = {
    "macro_f1": f1_score(y_test, test_pred, average="macro"),
    "accuracy": accuracy_score(y_test, test_pred),
    "report": classification_report(y_test, test_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0),
    "confusion_matrix": confusion_matrix(y_test, test_pred, labels=list(range(4))),
}
save_result("01_gradient_boosting", test_metrics_m1, val_f1_m1)
print(classification_report(y_test, test_pred, target_names=CLASS_NAMES, zero_division=0))

## Model 02 — mobilenet_v3_large

**Kenapa:** MobileNetV3-Large — arsitektur ringan; gambar secara visual sederhana (1 objek, background polos) jadi kemungkinan besar cukup, bukan under-powered.

In [ ]:
# Sub-Step M02.1
# Tujuan: Bangun & latih mobilenet_v3_large, evaluasi di test, simpan probabilitas untuk kandidat ensemble

model, head = build_model("mobilenet_v3_large", num_classes=4)
model, history, val_f1 = train_one_model(model, head, fit_loader, val_loader, device, "02_mobilenet_v3_large", lr_phase2=LR_PHASE2)
test_metrics = evaluate(model, test_loader, device)
save_result("02_mobilenet_v3_large", test_metrics, val_f1)
cnn_val_f1["02_mobilenet_v3_large"] = val_f1
cnn_test_proba["02_mobilenet_v3_large"] = predict_proba_torch(model, test_loader, device)
print(classification_report(test_metrics["y_true"], test_metrics["y_pred"], target_names=CLASS_NAMES, zero_division=0))
del model
torch.cuda.empty_cache()

## Model 03 — efficientnet_b0

**Kenapa:** EfficientNet-B0 — rasio akurasi/efisiensi terbaik di kelasnya untuk dataset kecil-menengah.

In [ ]:
# Sub-Step M03.1
# Tujuan: Bangun & latih efficientnet_b0, evaluasi di test, simpan probabilitas untuk kandidat ensemble

model, head = build_model("efficientnet_b0", num_classes=4)
model, history, val_f1 = train_one_model(model, head, fit_loader, val_loader, device, "03_efficientnet_b0", lr_phase2=LR_PHASE2)
test_metrics = evaluate(model, test_loader, device)
save_result("03_efficientnet_b0", test_metrics, val_f1)
cnn_val_f1["03_efficientnet_b0"] = val_f1
cnn_test_proba["03_efficientnet_b0"] = predict_proba_torch(model, test_loader, device)
print(classification_report(test_metrics["y_true"], test_metrics["y_pred"], target_names=CLASS_NAMES, zero_division=0))
del model
torch.cuda.empty_cache()

## Model 04 — resnet18

**Kenapa:** ResNet-18 — arsitektur paling dipahami & paling banyak tooling-nya, baseline CNN yang stabil.

In [ ]:
# Sub-Step M04.1
# Tujuan: Bangun & latih resnet18, evaluasi di test, simpan probabilitas untuk kandidat ensemble

model, head = build_model("resnet18", num_classes=4)
model, history, val_f1 = train_one_model(model, head, fit_loader, val_loader, device, "04_resnet18", lr_phase2=LR_PHASE2)
test_metrics = evaluate(model, test_loader, device)
save_result("04_resnet18", test_metrics, val_f1)
cnn_val_f1["04_resnet18"] = val_f1
cnn_test_proba["04_resnet18"] = predict_proba_torch(model, test_loader, device)
print(classification_report(test_metrics["y_true"], test_metrics["y_pred"], target_names=CLASS_NAMES, zero_division=0))
del model
torch.cuda.empty_cache()

## Model 05 — convnext_tiny

**Kenapa:** ConvNeXt-Tiny — inductive bias konvolusional modern, alternatif ResNet dengan risiko overfitting lebih terkendali dibanding ViT murni.

In [ ]:
# Sub-Step M05.1
# Tujuan: Bangun & latih convnext_tiny, evaluasi di test, simpan probabilitas untuk kandidat ensemble

model, head = build_model("convnext_tiny", num_classes=4)
model, history, val_f1 = train_one_model(model, head, fit_loader, val_loader, device, "05_convnext_tiny", lr_phase2=LR_PHASE2)
test_metrics = evaluate(model, test_loader, device)
save_result("05_convnext_tiny", test_metrics, val_f1)
cnn_val_f1["05_convnext_tiny"] = val_f1
cnn_test_proba["05_convnext_tiny"] = predict_proba_torch(model, test_loader, device)
print(classification_report(test_metrics["y_true"], test_metrics["y_pred"], target_names=CLASS_NAMES, zero_division=0))
del model
torch.cuda.empty_cache()

## Model 06 — DeiT-Tiny (pengganti ViT-Tiny/Small)

**Kenapa:** menangkap pola global (hubungan warna-bentuk keseluruhan bean) yang mungkin
terlewat CNN lokal. DeiT-Tiny (bukan ViT-Base bawaan torchvision yang 86M parameter --
terlalu besar untuk ~700 gambar train) dipilih karena didesain khusus lebih data-efficient.
**Catatan risiko:** ViT/DeiT tanpa inductive bias konvolusional lebih rakus data --
bandingkan hasilnya ke model 03/05 sebelum diadopsi, jangan jadi pilihan pertama untuk
dataset sekecil ini. LR fase-2 diperkecil (5e-5, bukan 3e-4) karena arsitektur transformer
umumnya lebih sensitif terhadap LR besar saat fine-tuning.

In [ ]:
# Sub-Step M06.1
# Tujuan: Bangun & latih DeiT-Tiny, evaluasi di test, simpan probabilitas untuk kandidat ensemble

model, head = build_model("deit_tiny", num_classes=4)
model, history, val_f1 = train_one_model(model, head, fit_loader, val_loader, device, "06_deit_tiny", lr_phase2=LR_PHASE2_VIT)
test_metrics = evaluate(model, test_loader, device)
save_result("06_deit_tiny", test_metrics, val_f1)
cnn_val_f1["06_deit_tiny"] = val_f1
cnn_test_proba["06_deit_tiny"] = predict_proba_torch(model, test_loader, device)
print(classification_report(test_metrics["y_true"], test_metrics["y_pred"], target_names=CLASS_NAMES, zero_division=0))
del model
torch.cuda.empty_cache()

## Model 07 — Two-Stage Hierarchical (tipe bean + status rusak terpisah)

**Kenapa:** implementasi langsung dari rekomendasi utama laporan EDA v3. Memecah problem
4-kelas yang secara internal tumpang tindih (defect = kemungkinan besar campuran ketiga
jenis lain yang rusak) jadi dua sub-problem yang masing-masing lebih homogen: (a) classifier
tipe bean (3-kelas, HANYA dilatih di sampel non-defect), (b) classifier biner rusak/tidak
(semua data). Prediksi akhir digabung balik: kalau (b) bilang "rusak" -> `defect`, kalau
tidak -> ambil hasil (a).

In [ ]:
# Sub-Step M07.1
# Tujuan: Siapkan sub-dataset tipe (non-defect saja) & rusak/tidak (semua data)

TYPE_LABEL_FN = lambda l: {"premium": 0, "peaberry": 1, "longberry": 2}[l]
DAMAGE_LABEL_FN = lambda l: 1 if l == "defect" else 0

fit_df_type = fit_df[fit_df["label"] != "defect"].reset_index(drop=True)
val_df_type = val_df[val_df["label"] != "defect"].reset_index(drop=True)

fit_type_loader = DataLoader(BeanDataset(fit_df_type, PREP_DIR, train_transform, label_fn=TYPE_LABEL_FN),
                              batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_type_loader = DataLoader(BeanDataset(val_df_type, PREP_DIR, eval_transform, label_fn=TYPE_LABEL_FN),
                              batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

fit_damage_loader = DataLoader(BeanDataset(fit_df, PREP_DIR, train_transform, label_fn=DAMAGE_LABEL_FN),
                                batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_damage_loader = DataLoader(BeanDataset(val_df, PREP_DIR, eval_transform, label_fn=DAMAGE_LABEL_FN),
                                batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Type sub-task: fit={len(fit_df_type)} val={len(val_df_type)} | Damage sub-task: fit={len(fit_df)} val={len(val_df)}")

In [ ]:
# Sub-Step M07.2
# Tujuan: Latih classifier tipe (3-kelas) & classifier rusak/tidak (biner)

type_model, type_head = build_model("efficientnet_b0", num_classes=3)
type_model, _, type_val_f1 = train_one_model(
    type_model, type_head, fit_type_loader, val_type_loader, device, "07a_type",
    class_names=["premium", "peaberry", "longberry"],
)

damage_model, damage_head = build_model("efficientnet_b0", num_classes=2)
damage_model, _, damage_val_f1 = train_one_model(
    damage_model, damage_head, fit_damage_loader, val_damage_loader, device, "07b_damage",
    class_names=["intact", "defect"],
)

In [ ]:
# Sub-Step M07.3
# Tujuan: Gabungkan kedua model jadi prediksi 4-kelas flat, evaluasi di test

def hierarchical_predict_fn(images):
    damage_pred = damage_model(images).argmax(dim=1).cpu().numpy()
    type_pred = type_model(images).argmax(dim=1).cpu().numpy()
    return np.where(damage_pred == 1, LABEL_TO_IDX["defect"], [TYPE_TO_FLAT[t] for t in type_pred])

val_metrics_m7 = evaluate_combined(hierarchical_predict_fn, val_loader, device)
test_metrics_m7 = evaluate_combined(hierarchical_predict_fn, test_loader, device)
save_result("07_hierarchical", test_metrics_m7, val_metrics_m7["macro_f1"],
            extra={"type_val_f1": type_val_f1, "damage_val_f1": damage_val_f1})
print(classification_report(test_metrics_m7["y_true"], test_metrics_m7["y_pred"], target_names=CLASS_NAMES, zero_division=0))
del type_model, damage_model
torch.cuda.empty_cache()

## Model 08 — Multi-Task Single-Backbone (1 backbone, 2 head)

**Kenapa:** alternatif model #7 yang lebih hemat data — berbagi representasi tingkat rendah
(edge, warna) antar dua task lewat satu backbone EfficientNet-B0, biasanya lebih
data-efficient daripada dua model penuh terpisah saat total data cuma ~700 gambar training.
Loss = loss_damage + loss_type, dengan loss_type di-mask supaya hanya berlaku untuk sampel
non-defect (sampel defect tidak punya label tipe yang valid).

In [ ]:
# Sub-Step M08.1
# Tujuan: DataLoader multi-task (damage + type sekaligus per sampel)

fit_mt_loader = DataLoader(MultiTaskDataset(fit_df, PREP_DIR, train_transform),
                            batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_mt_loader = DataLoader(MultiTaskDataset(val_df, PREP_DIR, eval_transform),
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
# Sub-Step M08.2
# Tujuan: Loop training khusus multi-task (loss gabungan, loss_type di-mask)

@torch.no_grad()
def _mt_val_f1(model, loader, device):
    model.eval()
    preds, labels_flat = [], []
    for images, damage_labels, type_labels in loader:
        images = images.to(device)
        out_damage, out_type = model(images)
        damage_pred = out_damage.argmax(dim=1).cpu().numpy()
        type_pred = out_type.argmax(dim=1).cpu().numpy()
        combined = np.where(damage_pred == 1, LABEL_TO_IDX["defect"], [TYPE_TO_FLAT[t] for t in type_pred])
        preds.extend(list(combined))
        dmg_np, typ_np = damage_labels.numpy(), type_labels.numpy()
        true_flat = np.where(dmg_np == 1, LABEL_TO_IDX["defect"], [TYPE_TO_FLAT.get(t, -1) for t in typ_np])
        labels_flat.extend(true_flat.tolist())
    return f1_score(labels_flat, preds, average="macro", zero_division=0)


def train_multitask(model, fit_loader, val_loader, device, model_name):
    model = model.to(device)
    ce_damage, ce_type = nn.CrossEntropyLoss(), nn.CrossEntropyLoss()
    best_state, best_val_f1, patience_counter = None, -1.0, 0

    def run_epoch(optimizer):
        model.train()
        for images, damage_labels, type_labels in fit_loader:
            images = images.to(device); damage_labels = damage_labels.to(device); type_labels = type_labels.to(device)
            optimizer.zero_grad()
            out_damage, out_type = model(images)
            loss = ce_damage(out_damage, damage_labels)
            mask = type_labels >= 0
            if mask.any():
                loss = loss + ce_type(out_type[mask], type_labels[mask])
            loss.backward()
            optimizer.step()

    # Phase 1: freeze backbone
    for p in model.backbone.parameters():
        p.requires_grad = False
    optimizer = torch.optim.AdamW(
        list(model.head_damage.parameters()) + list(model.head_type.parameters()),
        lr=LR_PHASE1, weight_decay=WEIGHT_DECAY,
    )
    for epoch in range(EPOCHS_PHASE1):
        run_epoch(optimizer)
        val_f1 = _mt_val_f1(model, val_loader, device)
        if val_f1 > best_val_f1:
            best_val_f1, best_state, patience_counter = val_f1, copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1

    # Phase 2: unfreeze all
    for p in model.parameters():
        p.requires_grad = True
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_PHASE2, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=SCHEDULER_PATIENCE)
    patience_counter = 0
    for epoch in range(EPOCHS_PHASE2):
        run_epoch(optimizer)
        val_f1 = _mt_val_f1(model, val_loader, device)
        scheduler.step(val_f1)
        if val_f1 > best_val_f1:
            best_val_f1, best_state, patience_counter = val_f1, copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"  [{model_name}] early stop phase-2 epoch {epoch} (val_macro_f1 terbaik={best_val_f1:.4f})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"[{model_name}] val_macro_f1 terbaik={best_val_f1:.4f}")
    return model, best_val_f1


mt_model = EfficientNetMultiTask()
mt_model, val_f1_m8 = train_multitask(mt_model, fit_mt_loader, val_mt_loader, device, "08_multitask")

In [ ]:
# Sub-Step M08.3
# Tujuan: Evaluasi model multi-task di test (loader flat 4-kelas biasa)

def multitask_predict_fn(images):
    out_damage, out_type = mt_model(images)
    damage_pred = out_damage.argmax(dim=1).cpu().numpy()
    type_pred = out_type.argmax(dim=1).cpu().numpy()
    return np.where(damage_pred == 1, LABEL_TO_IDX["defect"], [TYPE_TO_FLAT[t] for t in type_pred])

test_metrics_m8 = evaluate_combined(multitask_predict_fn, test_loader, device)
save_result("08_multitask", test_metrics_m8, val_f1_m8)
print(classification_report(test_metrics_m8["y_true"], test_metrics_m8["y_pred"], target_names=CLASS_NAMES, zero_division=0))
del mt_model
torch.cuda.empty_cache()

## Model 09 — EfficientNet-B0 + Label Smoothing + Sample-Weighting (Noise-Robust)

**Kenapa:** EDA menemukan label noise nyata tapi berbasis proxy (bukan ground truth pasti
-- lihat EDA v2 Section 12). Melatih dengan asumsi label 100% bersih itu berisiko; label
smoothing (0,1) + turunkan bobot sampel yang diduga mislabel (0,5, bukan exclude keras)
adalah pendekatan yang lebih robust terhadap ketidakpastian label ini. Daftar mislabel
**dihitung ulang** di sini (metode identik EDA v2: RandomForest + cross_val_predict
cluster-aware) pada fit pool 697 gambar yang sekarang, memakai fitur hand-crafted yang
sudah dihitung untuk Model 01 -- daftar asli dari run Kaggle EDA v2 tidak tersimpan
sebagai file, hanya tercetak di output notebook.

In [ ]:
# Sub-Step M09.1
# Tujuan: Hitung ulang proxy mistakenness (RandomForest OOF, cluster-aware) pada fit pool

from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier

sgkf_noise = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
rf_noise = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
proba_oof = cross_val_predict(rf_noise, X_fit, y_fit, cv=sgkf_noise,
                               groups=fit_df["cluster_id"].values, method="predict_proba")
true_proba = proba_oof[np.arange(len(y_fit)), y_fit]
max_proba = proba_oof.max(axis=1)
mistake_score = max_proba - true_proba
flagged_mask = mistake_score > 0.5
sample_weights_fit = np.where(flagged_mask, 0.5, 1.0)
print(f"Kandidat mislabel di fit pool: {flagged_mask.sum()} / {len(fit_df)}")

In [ ]:
# Sub-Step M09.2
# Tujuan: Latih EfficientNet-B0 dengan label_smoothing=0.1 + sample-weight kandidat mislabel

fit_ds_m9 = BeanDataset(fit_df, PREP_DIR, train_transform, weights=sample_weights_fit.tolist())
fit_loader_m9 = DataLoader(fit_ds_m9, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

criterion_m9 = nn.CrossEntropyLoss(label_smoothing=0.1, reduction="none")
model, head = build_model("efficientnet_b0", num_classes=4)
model, history, val_f1_m9 = train_one_model(
    model, head, fit_loader_m9, val_loader, device, "09_noise_robust", criterion=criterion_m9
)
test_metrics_m9 = evaluate(model, test_loader, device)
save_result("09_noise_robust", test_metrics_m9, val_f1_m9, extra={"n_flagged_mislabel": int(flagged_mask.sum())})
print(classification_report(test_metrics_m9["y_true"], test_metrics_m9["y_pred"], target_names=CLASS_NAMES, zero_division=0))
del model
torch.cuda.empty_cache()

## Model 10 — Ensemble: LightGBM (#01) + CNN Terbaik (#02-06)

**Kenapa:** fitur hand-crafted (bentuk/warna/tekstur) terbukti membawa sinyal kuat dan
mudah diinterpretasi; CNN belajar representasi yang mungkin overlap sebagian tapi juga
menangkap pola yang tidak terkodekan manual. Late-fusion (rata-rata probabilitas, bobot
0,5/0,5 default) sering menambah robustness pada dataset kecil.

In [ ]:
# Sub-Step M10.1
# Tujuan: Pilih CNN dengan val macro-F1 tertinggi di antara 02-06, gabungkan probabilitas dengan LightGBM

best_cnn_name = max(cnn_val_f1, key=cnn_val_f1.get)
print(f"CNN terbaik untuk ensemble: {best_cnn_name} (val macro-F1={cnn_val_f1[best_cnn_name]:.4f})")

lgb_test_proba = lgb_model.predict_proba(X_test)  # dari Model 01
best_cnn_test_proba = cnn_test_proba[best_cnn_name]

ensemble_proba = 0.5 * lgb_test_proba + 0.5 * best_cnn_test_proba
ensemble_pred = ensemble_proba.argmax(axis=1)

test_metrics_m10 = {
    "macro_f1": f1_score(y_test, ensemble_pred, average="macro"),
    "accuracy": accuracy_score(y_test, ensemble_pred),
    "report": classification_report(y_test, ensemble_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0),
    "confusion_matrix": confusion_matrix(y_test, ensemble_pred, labels=list(range(4))),
}
save_result("10_ensemble", test_metrics_m10, np.nan, extra={"ensemble_of": f"01_gradient_boosting + {best_cnn_name}"})
print(classification_report(y_test, ensemble_pred, target_names=CLASS_NAMES, zero_division=0))

## Ringkasan Akhir — Perbandingan 10 Model

Diurutkan berdasarkan `test_macro_f1` (metrik utama, sesuai `modeling-strategy.md` §3).
**Gate wajib**: model harus mengalahkan baseline 73% RandomForest dari EDA v2 Section 11
(fitur hand-crafted, CV cluster-aware) supaya dianggap valid -- kalau ada yang di bawah itu,
kemungkinan ada yang salah di pipeline, bukan sekadar "model kurang canggih".

In [ ]:
# Sub-Step 15.1
# Tujuan: Tabel perbandingan akhir, disimpan ke metadata/model_comparison.csv

BASELINE_MACRO_F1 = 0.73  # RandomForest + 11 fitur hand-crafted, EDA v2 Section 11 (CV cluster-aware)

comparison_df = pd.DataFrame(results.values()).sort_values("test_macro_f1", ascending=False).reset_index(drop=True)
comparison_df["beats_baseline"] = comparison_df["test_macro_f1"] > BASELINE_MACRO_F1

Path("metadata").mkdir(exist_ok=True)
comparison_df.to_csv("metadata/model_comparison.csv", index=False)

print(f"DRY_RUN={DRY_RUN} -- angka di bawah ini {'BELUM final (epoch kecil, hanya tes kode)' if DRY_RUN else 'adalah hasil full run'}")
print()
display_cols = ["model", "val_macro_f1", "test_macro_f1", "test_accuracy",
                 "test_recall_defect", "test_recall_longberry", "test_recall_peaberry", "test_recall_premium",
                 "beats_baseline"]
print(comparison_df[display_cols].round(4).to_string(index=False))